# *CARGA DEL DATASET*

In [29]:
import pandas as pd

df = pd.read_csv("sales_data_sample.csv", encoding="latin-1")

print(f"Filas: {df.shape[0]}")
print(f"Columnas: {df.shape[1]}")
print("\nNombres de columnas:")
print(df.columns.tolist())

Filas: 2823
Columnas: 25

Nombres de columnas:
['ORDERNUMBER', 'QUANTITYORDERED', 'PRICEEACH', 'ORDERLINENUMBER', 'SALES', 'ORDERDATE', 'STATUS', 'QTR_ID', 'MONTH_ID', 'YEAR_ID', 'PRODUCTLINE', 'MSRP', 'PRODUCTCODE', 'CUSTOMERNAME', 'PHONE', 'ADDRESSLINE1', 'ADDRESSLINE2', 'CITY', 'STATE', 'POSTALCODE', 'COUNTRY', 'TERRITORY', 'CONTACTLASTNAME', 'CONTACTFIRSTNAME', 'DEALSIZE']


In [30]:
df.head()

,ORDERNUMBER,QUANTITYORDERED,PRICEEACH,ORDERLINENUMBER,SALES,ORDERDATE,STATUS,QTR_ID,MONTH_ID,YEAR_ID,...,ADDRESSLINE1,ADDRESSLINE2,CITY,STATE,POSTALCODE,COUNTRY,TERRITORY,CONTACTLASTNAME,CONTACTFIRSTNAME,DEALSIZE
0,10107,30,95.70,2,2871.00,2/24/2003 0:00,Shipped,1,2,2003,...,897 Long Airport Avenue,NaN,NYC,NY,10022,USA,NaN,Yu,Kwai,Small
1,10121,34,81.35,5,2765.90,5/7/2003 0:00,Shipped,2,5,2003,...,59 rue de l'Abbaye,NaN,Reims,NaN,51100,France,EMEA,Henriot,Paul,Small
2,10134,41,94.74,2,3884.34,7/1/2003 0:00,Shipped,3,7,2003,...,27 rue du Colonel Pierre Avia,NaN,Paris,NaN,75508,France,EMEA,Da Cunha,Daniel,Medium
3,10145,45,83.26,6,3746.70,8/25/2003 0:00,Shipped,3,8,2003,...,78934 Hillside Dr.,NaN,Pasadena,CA,90003,USA,NaN,Young,Julie,Medium
4,10159,49,100.00,14,5205.27,10/10/2003 0:00,Shipped,4,10,2003,...,7734 Strong St.,NaN,San Francisco,CA,NaN,USA,NaN,Brown,Julie,Medium


# Normalizacion del *dataset*


In [31]:
# Normalizar nombres de columnas
df.columns = df.columns.str.strip().str.upper().str.replace(" ", "_")

# Verificar valores nulos
print("Valores nulos por columna:")
print(df.isnull().sum())

Valores nulos por columna:
ORDERNUMBER            0
QUANTITYORDERED        0
PRICEEACH              0
ORDERLINENUMBER        0
SALES                  0
ORDERDATE              0
STATUS                 0
QTR_ID                 0
MONTH_ID               0
YEAR_ID                0
PRODUCTLINE            0
MSRP                   0
PRODUCTCODE            0
CUSTOMERNAME           0
PHONE                  0
ADDRESSLINE1           0
ADDRESSLINE2        2521
CITY                   0
STATE               1486
POSTALCODE            76
COUNTRY                0
TERRITORY           1074
CONTACTLASTNAME        0
CONTACTFIRSTNAME       0
DEALSIZE               0
dtype: int64


# Verificar si hay caracteres especiales

In [32]:
# Aplicamos la búsqueda permitiendo signos comunes en nombres y direcciones
# Añadimos dentro del corchete: \. (punto), , (coma), - (guión) y ' (apóstrofe)
patron_realista = r'[^a-zA-Z0-9\sáéíóúÁÉÍÓÚñÑ\.,\-\']'

caracteres_por_columna = df.select_dtypes(include=['object']).apply(
    lambda col: col.str.contains(patron_realista, na=False, regex=True).sum()
)

print("Cantidad de caracteres especiales reales por columna:")
print(caracteres_por_columna)

Cantidad de caracteres especiales reales por columna:
ORDERDATE           2823
STATUS                 0
PRODUCTLINE            0
PRODUCTCODE         2823
CUSTOMERNAME         150
PHONE                997
ADDRESSLINE1         455
ADDRESSLINE2           0
CITY                   0
STATE                  0
POSTALCODE             0
COUNTRY                0
TERRITORY              0
CONTACTLASTNAME        0
CONTACTFIRSTNAME      32
DEALSIZE               0
dtype: int64


In [33]:
# Ver qué valores específicos están saltando en los nombres de los clientes
print(df[df['CUSTOMERNAME'].str.contains(r'[^a-zA-Z0-9\s]', na=False, regex=True)]['CUSTOMERNAME'].unique()[:10])

['Land of Toys Inc.' 'Toys4GrownUps.com' 'Corporate Gift Ideas Co.'
 'Technics Stores Inc.' 'Mini Wheels Co.' 'Australian Collectors, Co.'
 'Vitachrome Inc.' 'Tekni Collectables Inc.' 'Gift Depot Inc.'
 "Marta's Replicas Co."]


In [34]:
# 1. Ver qué nombres de clientes siguen saltando con el PATRÓN REALISTA
print("Clientes rebeldes:")
print(df[df['CUSTOMERNAME'].str.contains(patron_realista, na=False, regex=True)]['CUSTOMERNAME'].unique()[:10])

# 2. Ver qué nombres de contactos siguen saltando con el PATRÓN REALISTA
print("\nContactos rebeldes:")
print(df[df['CONTACTFIRSTNAME'].str.contains(patron_realista, na=False, regex=True)]['CONTACTFIRSTNAME'].unique()[:10])

Clientes rebeldes:
['Saveley & Henriot, Co.' 'Amica Models & Co.' 'Auto Assoc. & Cie.'
 'Handji Gifts& Co' 'Cruz & Sons Co.' 'Boards & Toys Co.']

Contactos rebeldes:
['Mart¡n']


In [35]:
# 1. Corregimos el error de encoding en todo el dataset (cambiar '¡' por 'í')
# Esto arreglará 'Mart¡n' y cualquier otro nombre afectado
df = df.replace('¡', 'í', regex=True)

# 2. Actualizamos el patrón realista para PERMITIR el ampersand '&'
patron_realista = r'[^a-zA-Z0-9\sáéíóúÁÉÍÓÚñÑ\.,\-\'\&]'

# 3. Volvemos a calcular el conteo para verificar
caracteres_por_columna = df.select_dtypes(include=['object']).apply(
    lambda col: col.str.contains(patron_realista, na=False, regex=True).sum()
)

print("Cantidad de caracteres especiales final:")
print(caracteres_por_columna)

Cantidad de caracteres especiales final:
ORDERDATE           2823
STATUS                 0
PRODUCTLINE            0
PRODUCTCODE         2823
CUSTOMERNAME           0
PHONE                997
ADDRESSLINE1         455
ADDRESSLINE2           0
CITY                   0
STATE                  0
POSTALCODE             0
COUNTRY                0
TERRITORY              0
CONTACTLASTNAME        0
CONTACTFIRSTNAME       0
DEALSIZE               0
dtype: int64


In [36]:
#Tipos de datos en df
print("Tipos de datos por columnas: ")
print(df.dtypes)

# Estadisticas generales
print("Estadisticas de columnas numericas: ")
df.describe()

Tipos de datos por columnas: 
ORDERNUMBER           int64
QUANTITYORDERED       int64
PRICEEACH           float64
ORDERLINENUMBER       int64
SALES               float64
ORDERDATE            object
STATUS               object
QTR_ID                int64
MONTH_ID              int64
YEAR_ID               int64
PRODUCTLINE          object
MSRP                  int64
PRODUCTCODE          object
CUSTOMERNAME         object
PHONE                object
ADDRESSLINE1         object
ADDRESSLINE2         object
CITY                 object
STATE                object
POSTALCODE           object
COUNTRY              object
TERRITORY            object
CONTACTLASTNAME      object
CONTACTFIRSTNAME     object
DEALSIZE             object
dtype: object
Estadisticas de columnas numericas: 


,ORDERNUMBER,QUANTITYORDERED,PRICEEACH,ORDERLINENUMBER,SALES,QTR_ID,MONTH_ID,YEAR_ID,MSRP
count,2823.000000,2823.000000,2823.000000,2823.000000,2823.000000,2823.000000,2823.000000,2823.00000,2823.000000
mean,10258.725115,35.092809,83.658544,6.466171,3553.889072,2.717676,7.092455,2003.81509,100.715551
std,92.085478,9.741443,20.174277,4.225841,1841.865106,1.203878,3.656633,0.69967,40.187912
min,10100.000000,6.000000,26.880000,1.000000,482.130000,1.000000,1.000000,2003.00000,33.000000
25%,10180.000000,27.000000,68.860000,3.000000,2203.430000,2.000000,4.000000,2003.00000,68.000000
50%,10262.000000,35.000000,95.700000,6.000000,3184.800000,3.000000,8.000000,2004.00000,99.000000
75%,10333.500000,43.000000,100.000000,9.000000,4508.000000,4.000000,11.000000,2004.00000,124.000000
max,10425.000000,97.000000,100.000000,18.000000,14082.800000,4.000000,12.000000,2005.00000,214.000000


# Guardar el CSV

In [37]:
df.to_csv("data_sample.csv", index=False)
print("\n Dataset limpio guardado como data_sample.csv")


 Dataset limpio guardado como data_sample.csv
